In [ ]:
from datetime import datetime as dt

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from ucimlrepo import fetch_ucirepo 

## 1. データ読み込み

In [ ]:
# fetch dataset 
online_retail = fetch_ucirepo(id=352) 

In [ ]:
df_raw = online_retail.data.original

In [ ]:
df_raw.head()

## 2. 元データの前処理

In [ ]:
df_data = df_raw.copy()

In [ ]:
df_data.dtypes

In [ ]:
df_data.isnull().sum()

In [ ]:
# Recencyの基準日をデータの最終日の翌日にする
date_format = "%m/%d/%Y %H:%M"

recency_day= pd.to_datetime(
    df_data["InvoiceDate"],
    format=date_format
).max() + pd.Timedelta(days=1)

print(recency_day)

### 2.1 CustomerIDのnullを削除

In [ ]:
def filter_null(df):
    
    df = df[df["CustomerID"].notnull()]

    return df.reset_index(drop=True)

In [ ]:
# 動作確認
df_tmp = filter_null(df=df_data)

In [ ]:
df_tmp["CustomerID"].isnull().sum()

### 2.2 InvoiceNoのキャセル分を削除

In [ ]:
def filter_cancelled(df):

    # 堅牢性のため、大文字として扱う
    # 元データがイギリスのデータであり、日本語ではないため、全角半角変換は入れない
    df["cancelled_flag"] = df["InvoiceNo"].astype(str).str.upper().str.startswith("C")

    return df[~df["cancelled_flag"]].reset_index(drop=True)

In [ ]:
# 動作確認
df_tmp = filter_cancelled(df=df_data)

In [ ]:
df_tmp["CustomerID"].astype(str).str.upper().str.startswith("C").sum()

### 2.3 UnitPriceとQuantityの負の値の削除

In [ ]:
def filter_minus(df):
    
    df = df[df["Quantity"]>0]
    df = df[df["UnitPrice"]>0]

    return df

In [ ]:
# 動作確認
df_tmp = filter_minus(df=df_data)

In [ ]:
len(df_tmp[df_tmp["Quantity"]<0])

In [ ]:
len(df_tmp[df_tmp["UnitPrice"]<0])

### 2.4 InvoiceDateをdatetime型に変換

In [ ]:
def convert_data(df):

    date_format = "%m/%d/%Y %H:%M"

    df["InvoiceDate"] = pd.to_datetime(
        df["InvoiceDate"],
        format=date_format
    )

    return df.reset_index(drop=True)

In [ ]:
df_tmp = convert_data(df=df_data)

In [ ]:
df_tmp.head(3)

### 2.5 単価×数量の計算

In [ ]:
def calc_linetotal(df):

    df["line_total"] = df["Quantity"] * df["UnitPrice"]

    return df

In [ ]:
df_tmp = calc_linetotal(df=df_data)

In [ ]:
df_tmp.head(3)

### 2.6 前処理の実施

In [ ]:
def preprocessing(df):
    
    df = filter_null(df=df)
    df = filter_cancelled(df=df)
    df = filter_minus(df=df)
    df = convert_data(df=df)
    df = calc_linetotal(df=df)

    return df
    

In [ ]:
df_filtered = preprocessing(df=df_data)

In [ ]:
df_filtered.head(10)

## 3. RFMの算出

In [ ]:
list_use_cols = ["CustomerID", "InvoiceNo", "InvoiceDate", "line_total"]

### 3.1 Recency（最終購入日・直近の購買）

In [ ]:
df_filtered[list_use_cols]

In [ ]:
df_r = df_filtered[list_use_cols].groupby("CustomerID")["InvoiceDate"].max().to_frame()

In [ ]:
df_r.rename(columns={"InvoiceDate": "last_purchase_date"}, inplace=True)

In [ ]:
df_r.head(3)

### 3.2 Frequency（購入頻度・来店頻度）

In [ ]:
df_f = (
    df_filtered
    .groupby("CustomerID")["InvoiceNo"]
    .nunique()
    .to_frame(name="frequency")
)

In [ ]:
df_f.rename(columns={"InvoiceNo": "frequency"}, inplace=True)

In [ ]:
df_f.head(3)

### 3.3 Monetary（購入金額）

In [ ]:
df_m = df_filtered[list_use_cols].groupby("CustomerID")["line_total"].sum().to_frame()


In [ ]:
df_m.rename(columns={"line_total": "monetary"}, inplace=True)

In [ ]:
df_m.head(3)

### 3.4 RFMの結合

In [ ]:
df_rfm = df_r

In [ ]:
df_rfm = df_rfm.join(df_f)

In [ ]:
df_rfm = df_rfm.join(df_m)

In [ ]:
df_rfm.head()

In [ ]:
# joinが失敗してデータが落ちてないか確認
df_rfm.isnull().sum()

## 4. RFM分析

In [ ]:
df_rfm["recency"] = (recency_day - df_rfm["last_purchase_date"]).dt.days

In [ ]:
df_rfm

### 4.1 分布を確認

In [ ]:
sns.histplot(
    data=df_rfm,
    x="recency"
)

plt.ylabel("Number of customers")
plt.title("Histogram of Recency")
plt.show()

In [ ]:
sns.histplot(
    data=df_rfm,
    x="monetary"
)

plt.ylabel("Number of customers")
plt.title("Histogram of Monetary")
plt.show()

# 一部の大きな値がある

In [ ]:
sns.histplot(
    data=df_rfm,
    x="frequency"
)

plt.ylabel("Number of customers")
plt.title("Histogram of Frequency")
plt.show()

# 一部の大きな値がある

### 4.2 分布を変換

In [ ]:
scaler = StandardScaler()

In [ ]:
# recencyを標準化
df_rfm[["scaled_recency"]] = scaler.fit_transform(df_rfm[["recency"]])

In [ ]:
# 対数変換→標準化のパイプラインを定義

pipe = Pipeline([
    ("log", FunctionTransformer(func=np.log1p, inverse_func=np.expm1)),
    ("scaler", StandardScaler())
])

In [ ]:
# monetataryを対数変換してから、標準化

df_rfm["scaled_log_monetary"] = pipe.fit_transform(df_rfm[["monetary"]])


In [ ]:
# frequencyを対数変換してから、標準化

df_rfm["scaled_log_frequency"] = pipe.fit_transform(df_rfm[["frequency"]])

In [ ]:
df_rfm

### 4.3 変換後の分布の確認

In [ ]:
sns.histplot(
    data=df_rfm,
    x="scaled_recency"
)

plt.ylabel("Number of customers")
plt.title("Histogram of scaled_recency")
plt.show()

In [ ]:
sns.histplot(
    data=df_rfm,
    x="scaled_log_monetary"
)

plt.ylabel("Number of customers")
plt.title("Histogram of scaled_log_monetary")
plt.show()

In [ ]:
sns.histplot(
    data=df_rfm,
    x="scaled_log_frequency"
)

plt.ylabel("Number of customers")
plt.title("Histogram of scaled_log_frequency")
plt.show()

### 4.4 エルボー法でクラスター数の確認

In [ ]:
# エルボー法用に、クラスター数を流動的に指定してインスタンスを作る

list_feature_cols = ["scaled_recency", "scaled_log_monetary", "scaled_log_frequency"]
sse = []
k_range = range(3, 11)  # クラスタ数のレンジ。

for k in k_range:
    model = KMeans(n_clusters=k, n_init=10, random_state=42)
    model.fit(df_rfm[list_feature_cols])
    sse.append(model.inertia_)  # 各KでのSSE（距離の二乗和）を記録

# エルボー法のグラフを描画
plt.figure(figsize=(8, 5))
plt.plot(k_range, sse, marker="o", color="b", linestyle="-")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("SSE (Inertia)")
plt.title("Elbow Method for Optimal K")
plt.grid(True)
plt.show()

### 4.5 kmeansでクラスタに分類

In [ ]:
# インスタンス作成
n_clusters = 4

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init="auto")

In [ ]:

labels = kmeans.fit_predict(df_rfm[list_feature_cols])

In [ ]:
df_rfm["label_kmeans"] = labels

In [ ]:
df_rfm.groupby("label_kmeans").size()

### 4.6 クラスタリング結果の確認

In [ ]:
df_rfm.head(3)

In [ ]:
df_rfm[["recency", "frequency", "monetary"]][df_rfm["label_kmeans"]==0].describe()

In [ ]:
df_rfm[["recency", "frequency", "monetary"]][df_rfm["label_kmeans"]==1].describe()

In [ ]:
df_rfm[["recency", "frequency", "monetary"]][df_rfm["label_kmeans"]==2].describe()

In [ ]:
df_rfm[["recency", "frequency", "monetary"]][df_rfm["label_kmeans"]==3].describe()

In [ ]:
# 3次元グラフで見てみる

dict_label_to_cluster_name ={
    0:"Regular Customers",
    1:"Loyal Customers",
    2:"VIP Customers",
    3:"At-Risk Customers"
}


fig = plt.figure(figsize=(7, 5), layout="tight")
ax = fig.add_subplot(projection="3d")

for label in np.arange(0, n_clusters):

    df = df_rfm[df_rfm["label_kmeans"]==label]

    x = df["scaled_recency"]
    y = df["scaled_log_monetary"]
    z = df["scaled_log_frequency"]

    ax.scatter(x, y, z, label=f"{dict_label_to_cluster_name[label]}")

ax.set_xlabel("Scaled Recency")
ax.set_ylabel("Scaled Log(Monetary)")
ax.set_zlabel("Scaled Log(Frequency)")
ax.set_box_aspect(None, zoom=0.85)

plt.legend()
plt.title(f"Customer Segments based on RFM Analysis")
plt.show()